<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/5%EC%9B%9429%EC%9D%BC%EB%8F%84%EC%84%9C%EA%B4%80%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn gradio sqlalchemy

In [6]:
import gradio as gr
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

# 1. DB 설정 (SQLite)
engine = create_engine("sqlite:///./library.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class DBBook(Base):
    __tablename__ = "books"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String)
    author = Column(String)
    borrower = Column(String, nullable=True)
    is_borrowed = Column(Boolean, default=False)

Base.metadata.create_all(bind=engine)

# 초기 데이터
def init_data():
    db = SessionLocal()
    if not db.query(DBBook).first():
        db.add_all([
            DBBook(id=1, title="파이썬의 마법", author="반 귀도", is_borrowed=False),
            DBBook(id=2, title="데이터베이스의 이해", author="SQL 마스터", is_borrowed=False),
            DBBook(id=3, title="코딩의 정석", author="알고리즘", is_borrowed=False)
        ])
        db.commit()
    db.close()
init_data()

# 기능함수
def get_library_html():
    db = SessionLocal()
    books = db.query(DBBook).all()
    db.close()
    html = "<div style='background:#f9f7f2; padding:20px; border-radius:10px; border:1px solid #dcdcdc;'>"
    html += "<h3 style='color:#5d4037;'>📚 도서 대출 현황판</h3><hr>"
    for b in books:
        status = "🔴 대출중" if b.is_borrowed else "🟢 대출가능"
        color = "#c62828" if b.is_borrowed else "#2e7d32"
        html += f"<div style='margin-bottom:10px; padding:10px; background:white; border-left:5px solid {color};'>"
        html += f"<strong>[{b.id}] {b.title}</strong> | 저자: {b.author} | <span style='color:{color}'>{status}</span>"
        if b.borrower: html += f" | 대출자: <b>{b.borrower}</b>"
        html += "</div>"
    html += "</div>"
    return html

def action_book(action, book_id, name):
    db = SessionLocal()
    b = db.query(DBBook).get(int(book_id))
    if not b: return "⚠️ 존재하지 않는 책입니다.", get_library_html()

    if action == "대출":
        if b.is_borrowed: return "⚠️ 이미 대출 중입니다.", get_library_html()
        b.is_borrowed = True; b.borrower = name
    else:
        if not b.is_borrowed: return "⚠️ 이미 반납된 책입니다.", get_library_html()
        b.is_borrowed = False; b.borrower = None

    db.commit(); db.close()
    return f"✅ '{b.title}' {action} 처리가 완료되었습니다.", get_library_html()

# 디자인 UI
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🏛️ 디지털 도서관 관리 시스템")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📝 도서 조작")
            id_in = gr.Number(label="도서 고유 ID", precision=0)
            name_in = gr.Textbox(label="회원 성함 (대출 시)")

            with gr.Row():
                btn_b = gr.Button("대출하기", variant="primary")
                btn_r = gr.Button("반납하기", variant="secondary")

            output_msg = gr.Markdown("상태 알림")

        with gr.Column(scale=2):
            display = gr.HTML(value=get_library_html())
            btn_ref = gr.Button("🔄 현황판 새로고침")

    btn_b.click(lambda i, n: action_book("대출", i, n), [id_in, name_in], [output_msg, display])
    btn_r.click(lambda i: action_book("반납", i, ""), [id_in], [output_msg, display])
    btn_ref.click(get_library_html, None, display)

demo.launch(share=True)

/tmp/ipykernel_1783/1123412975.py:9: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()
/tmp/ipykernel_1783/1123412975.py:67: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0dde79ae40b5100869.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
